# Automatic Encoding for the Iceberg Code 

We have seen in the [Steane code example](https://docs.quantinuum.com/guppy/ft/examples/steane_encoding.html) that Guppy FT provides automatic encoding for Guppy programs. In this demonstration, we will show automatic encoding for the $[[k+2, k, 2]]$ Iceberg code. The Iceberg code is distance $2$ so it can detect (but not correct) an arbitrary single qubit error. Once an error is possible to exit the program early before all of the gates are executed.

Unlike Steane, encoding programs with the Iceberg code requires the use of Nexus console - Quantinuum's cloud platform. As Nexus console is required, we will need the [qnexus](https://github.com/Quantinuum/qnexus) client package. The workflow is comprised of three steps.

1. Write a computational program in Guppy. A computational program can be written independently of QEC or fault-tolerance. We then compile this program. 
2. Secondly, define some Iceberg encoding parameters via the {py:class}`~guppyft.code.iceberg.encode.IcebergBuilder` class. These parameters will include the number of logical qubits in a block, the number of blocks and a policy for inserting quantum error detection (QED) cycles.
3. Finally, upload the annotated program to Nexus console. Compilation to the physical gates will happen serverside. The program will then be executed on a quantum device or simulator.

## Setup

The `qnexus` Python package can be installed as follows
```shell
pip install qnexus
```

In addition, we will also need a Nexus console account. If you don't already have access to Nexus console, it is possible to apply for access at https://nexus.quantinuum.com/signup.

Once we have the `qnexus` package installed, we can log in to Nexus console and create a project to keep track of all of the jobs we run.

In [ ]:
import qnexus as qnx

qnx.login()

project = qnx.projects.get_or_create(name="iceberg_project")
qnx.context.set_active_project(project)

## Write a computational program

We can now define our computational program using Guppy. For demonstration purposes, we will show a "mirror "program where we apply an $n$ qubit Quantum Fourier Transform ($QFT$) followed by a layer of Hadamard gates. The expected result in the absence of error would be the $|0\rangle^{\otimes n}$. This is true as the action of $QFT$ on the $|0\rangle^{\otimes n}$ state is the same as the Hadamard transform (The two transformations are self-inverse).

$$
QFT |0 \rangle ^{\otimes n} = H^{\otimes n}|0\rangle^{\otimes{n}}
$$

The $n$ qubit $QFT$ can be implemented with $n$ Hadamard gates and $\frac{n(n-1)}{2}$ controlled rotations. We choose this contrived mirror program because the quadratic number of two qubit gates means that the program is deep enough for errors to occur if we run it with a noise model. This will serve to demonstrate a QED policy (see next section). Given the mirrored structure of the program, it's also very easy to see which shots give the expected result of $|0\rangle^{\otimes n}$ and which shots are discarded after an error is detected.

In [ ]:
from guppylang import guppy
from guppylang.std.quantum import h, crz, qubit, rz
from guppylang.std.angles import pi, angle
from guppylang.std.mem import mem_swap


@guppy
def cphase(control: qubit, target: qubit, theta: angle) -> None:
    rz(control, theta / 2)
    crz(control, target, theta)


@guppy
def qft[n: nat](qs: array[qubit, n]) -> None:
    for i in range(n):
        h(qs[i])
        for j in range(i + 1, n):
            cphase(qs[j], qs[i], pi / 2 ** (j - i))

    # Reverse qubit order with swaps
    for k in range(n // 2):
        mem_swap(qs[k], qs[n - k - 1])

We also need a `main` function which defines the entrypoint to our quantum program. inside this `main` function we first allocate some qubits before applying the $QFT$ followed by a layer of Hadamard gates. Finally, we measure all of the qubits.

In [ ]:
from guppylang.std.platform import output
from guppylang.std.quantum import measure_array, collect_measurements
from guppylang.defs import GuppyFunctionDefinition


def build_qft_mirror_program(n_qubits: int) -> GuppyFunctionDefinition:
    @guppy
    def main() -> None:
        qs = array(qubit() for _ in range(n_qubits))
        qft(qs)
        for i in range(len(qs)):
            h(qs[i])

        measurements = measure_array(qs)
        output("m", collect_measurements(measurements))

    return main

We can now create 6 qubit instance of our QFT program and compile it to the HUGR intermediate representation.

In [ ]:
n_qft_qubits = 6

qft_main = build_qft_mirror_program(n_qft_qubits)
pkg = qft_main.compile()

## Define Iceberg encoding parameters

Now that we have our computational program, we can now use the {py:class}`~guppyft.code.iceberg.encode.IcebergBuilder` to specify some key parameters of the Iceberg code. 

The {py:class}`~guppyft.code.iceberg.encode.IcebergBuilder` should be configured with an {py:class}`~guppyft.code.iceberg.encode.IcebergQEDPolicy` - a policy for when quantum error detection (QED) cycles can be applied. Guppy FT allows users to associate costs with different logical operations. These costs can then be accumulated as the program runs and a QED cycle is applied whenever a user defined threshold is exceeded.


For our policy we will choose to assign a equal cost of `1.0` to Hadamard, CX and ZZPhase gates. A QED cycle will be performed after a threshold value of 10 is met. Note that the controlled rotation gates in the $QFT$ implementation will be compiled in terms of ZZPhase and single qubit gates.

In [ ]:
from guppyft.code.iceberg.encode import IcebergBuilder, IcebergQEDPolicy


qed_policy = IcebergQEDPolicy()

qed_policy.costs.cx_all = 1.0
qed_policy.costs.h_all = 1.0
qed_policy.costs.zz_phase_intrablock = 1.0

# Insert a QED cycle after 10 CX gates are applied
qed_policy.threshold = 10

Once we have defined a policy, we can use the {py:class}`~guppyft.code.iceberg.encode.IcebergBuilder` we have defined to annotate our compiled Guppy program with some metadata which stores all of the encoding parameters that we have specified. Computational to physical compilation can then be done serverside.

For this example we are going to our encode our 6 qubit $QFT$ mirror program in a single $[[8, 6, 2]]$ iceberg block.

In [ ]:
# Specify the block size (number of logical qubits) and number of blocks
block_size = 6
n_blocks = 1


builder = IcebergBuilder.new(block_size=block_size, n_blocks=n_blocks).with_qed_policy(
    qed_policy
)
instance = builder.build()

# Annotate block_size, n_blocks, qed_policy and other parameters onto the compiled program.
instance.annotate(pkg)

## Upload to Nexus console for execution

We can now upload our $QFT$ program to the cloud using the `qnexus` client for compilation and execution. This program is now annotated with all of the iceberg parameters that we specified in the previous section. 

In [ ]:
import uuid

name = f"iceberg-nexus-{uuid.uuid4()}"
iceberg_hugr_ref = qnx.hugr.upload(hugr_package=pkg, name=name)

We can now define a configuration for the `Helios-1E` remote emulator. This emulator uses a physics informed noise model to emulate the noise present in the Quantinuum Helios device.

In [ ]:
config = qnx.HeliosConfig(
    system_name="Helios-1E", emulator_config=qnx.models.HeliosEmulatorConfig()
)

Finally, we can submit an execute job to run on the `Helios-1E` simulator by passing a reference to our iceberg program to the `qnx.start_execute_job` method. Note how we have to specify the `max_cost` in HQCs and also specify a total number of physical qubits to be used in the simulation.

Note that in general it is not straightforward to predict the number of qubits needed by the physical program. To execute the encoded program without ancilla qubits used for error detection we need at least 8 qubits for the $[[8, 6, 2]]$ block. We choose to set an upper bound of 16 physical qubits to allow plenty of ancilla qubits for error detection during runtime. We will execute this encoded $QFT$ program on the noisy emulator for 500 shots.


```{note}
Before running encoded programs it's highly recommended to test the program with the coinflip or classical replay simulators. This can help determine the number of physical qubits used at runtime and potentially detect issues which could otherwise manifest as runtime errors.

See the [Nexus console docs](https://docs.quantinuum.com/nexus/trainings/notebooks/basics/selene_examples.html#coinflip-simulation) for information on how to use the coinflip and classical replay simulations. 
```

In [ ]:
ref_execute_job = qnx.start_execute_job(
    programs=[iceberg_hugr_ref],
    n_shots=[500],
    backend_config=config,
    max_cost=1000,
    name=f"Try Iceberg support QFT-(n_logical_qubits={n_qft_qubits}, block_size={block_size})",
    n_qubits=16,
)

In [ ]:
results = qnx.jobs.results(ref_execute_job)
result_ref = results[0]
sys_result = result_ref.download_result()

In [ ]:
# TODO: once exit messages are available via qnexus, discuss them here.

all_zero_counts = sys_result.collated_counts()[("m", "000000"),]
print(all_zero_counts)

In [ ]:
exit_counts = sys_result.collated_counts()[()]
print(exit_counts)